# Qwen3-1.7B v2 parser eval (Kaggle)

Evaluates a LoRA-tuned `unsloth/Qwen3-1.7B-bnb-4bit` adapter against the v3 held-out eval set with the v2 schema scoring rules. Mirrors `evaluate_finetune.py` from the local repo, with Kaggle-friendly paths and a notebook-style config cell instead of CLI flags.

## Prerequisites

1. **Eval dataset on Kaggle.** Upload `eval_finetune_dataset_v3_schema_frozen/heldout_cases.jsonl` as a Kaggle dataset (or use the small/medium variants you generated with `--total 50` / `--total 100`). Note the slug â€” it becomes part of `/kaggle/input/<slug>/`.
2. **Adapter on Kaggle.** Either:
   - Add the fine-tune notebook's saved version output as an input dataset to this notebook (right sidebar â†’ "+ Add Data" â†’ "Notebook Output Files" â†’ pick the version of `finetune_qwen3_1p7b.ipynb`); or
   - Upload the `lora_adapter/` folder as its own Kaggle dataset.
3. **Add both as inputs.** Right sidebar â†’ "+ Add Data" â†’ search â†’ Add for each.
4. **Pick a GPU accelerator.** Right sidebar â†’ Settings â†’ Accelerator â†’ `GPU T4 x2` or `GPU P100`.
5. **Enable Internet.** Right sidebar â†’ Settings â†’ Internet â†’ On.

## What it does

Loads the held-out cases, loads the base model + your adapter via `peft`, runs greedy inference on each case with the v2 chat-template framing (system prompt + `Today: <anchor_date>` per row), parses the model's JSON, and scores it with per-row schema routing â€” v2 rows score on `disposition` / `clarify_reason` / `clarify_options` / `reason_code` plus the existing per-lane metrics; legacy v1 rows score on legacy metrics only. Predictions and a summary land under `/kaggle/working/eval_run/`.

## Cells

1. CONFIG â€” paths and inference knobs (edit me).
2. Install packages.
3. Imports + GPU info.
4. Helpers (chat template, JSON parsing, score_prediction with v2 routing, summarize_rows).
5. Load the eval cases.
6. Load base model + adapter.
7. Run inference + score.
8. Save predictions + summary, print headline metrics.
9. Show top exact-match failures.

In [ ]:
# ---------- CONFIG (edit these for your setup) ----------
# Eval dataset jsonl. Adjust to your Kaggle dataset slug + filename.
DATASET = "/kaggle/input/eval-finetune-v3-schema-frozen/heldout_cases.jsonl"

# Adapter folder (peft format). Two common locations:
#   1. From a previously committed finetune_qwen3_1p7b.ipynb run (pulled in as input):
FINETUNED_MODEL = "/kaggle/input/qwen3-1p7b-v2-lora-adapter/lora_adapter"
#   2. Re-using the SAME kernel (back-to-back run after finetune_qwen3_1p7b.ipynb):
# FINETUNED_MODEL = "/kaggle/working/unsloth_qwen3_parser_run/lora_adapter"

FINETUNED_BASE_MODEL = "unsloth/Qwen3-1.7B-bnb-4bit"

# Optional: also score the BASE model alongside the fine-tune (slower; comment out for fine-tune-only).
BASE_MODEL = None  # e.g. "unsloth/Qwen3-1.7B-bnb-4bit"

OUTPUT_DIR = "/kaggle/working/eval_run"

MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 256

# Smoke-run a subset before committing the full eval. None = run all rows.
LIMIT = None  # e.g. 20 for a quick check

HF_TOKEN = None  # or wire from Kaggle Secrets below

SYSTEM_PROMPT = """You are a parser for a tag-first personal data app.
Return JSON only.
Do not add markdown.
Do not add explanations.
Do not add extra keys.
Use null for missing values.
Follow the schema shown by the examples exactly."""

# Optional: pull HF_TOKEN from Kaggle Secrets instead of pasting in plain text.
# from kaggle_secrets import UserSecretsClient
# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# ---------- INSTALL ----------
import sys, subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])

pip_install(
    "unsloth",
    "unsloth_zoo",
    "datasets",
    "transformers",
    "accelerate",
    "peft",
    "bitsandbytes",
)

In [ ]:
# ---------- IMPORTS + GPU info ----------
import gc
import json
from pathlib import Path

import unsloth  # noqa: F401  -- import before torch on some Kaggle envs
import torch
from peft import PeftModel
from unsloth import FastLanguageModel

print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ---------- HELPERS ----------
def build_system_prompt(anchor_date):
    """Append `Today: <YYYY-MM-DD>` to the system prompt for v2 rows. v1 rows
    (no anchor_date) keep the historical framing byte-identical."""
    if anchor_date:
        return f"{SYSTEM_PROMPT}\n\nToday: {anchor_date}"
    return SYSTEM_PROMPT


def load_cases(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    cases = []
    legacy_schema_rows = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if "id" not in row or "input" not in row or "expected" not in row:
                raise ValueError(f"Missing required fields in {path}:{line_no}")
            expected = row["expected"]
            if any(legacy_key in expected for legacy_key in ("date_refs", "date_text", "due_date_text")):
                legacy_schema_rows += 1
            cases.append(row)
    if legacy_schema_rows:
        print(f"Warning: {legacy_schema_rows} eval rows look like pre-v1 legacy schema rows.")
    return cases


def expected_has_any(expected, *keys):
    return any(key in expected for key in keys)


def build_user_text(case):
    user_text = case["input"].strip()
    if "context" in case:
        context_text = json.dumps(
            case["context"], ensure_ascii=False, separators=(",", ":"),
        )
        user_text = (
            "Previous structured query context:\n"
            f"{context_text}\n\n"
            "User input:\n"
            f"{user_text}"
        )
    return user_text


def normalize_json(value):
    if isinstance(value, dict):
        return {key: normalize_json(value[key]) for key in sorted(value)}
    if isinstance(value, list):
        return [normalize_json(item) for item in value]
    return value


def safe_json_loads(text):
    text = text.strip()
    if not text:
        return None, "empty_output"
    try:
        return json.loads(text), None
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        try:
            return json.loads(candidate), None
        except Exception as exc:
            return None, f"json_parse_error: {exc}"
    return None, "no_json_object_found"


def compare_record_field(expected_records, predicted_records, field_name):
    if not isinstance(predicted_records, list):
        return False
    if len(expected_records) != len(predicted_records):
        return False
    expected_values = [r.get(field_name) for r in expected_records]
    predicted_values = [r.get(field_name) for r in predicted_records]
    return expected_values == predicted_values


def compare_query_range(expected, predicted, start_key, end_key):
    if start_key not in expected and end_key not in expected:
        return None
    return predicted.get(start_key) == expected.get(start_key) and predicted.get(end_key) == expected.get(end_key)


def score_prediction(expected, predicted):
    metrics = {
        "valid_json": predicted is not None,
        "exact_match": False,
        "task_match": None,
        "lane_or_domain_match": None,
        "disposition_match": None,
        "reason_code_match": None,
        "clarify_reason_match": None,
        "clarify_options_match": None,
        "intent_match": None,
        "record_count_match": None,
        "ledger_action_match": None,
        "amounts_match": None,
        "weight_values_match": None,
        "write_dates_match": None,
        "unit_text_match": None,
        "query_date_range_match": None,
        "compare_range_match": None,
        "filters_match": None,
        "limit_match": None,
        "query_text_match": None,
        "inherit_context_match": None,
    }
    if predicted is None:
        return metrics

    metrics["exact_match"] = normalize_json(predicted) == normalize_json(expected)
    metrics["task_match"] = predicted.get("task") == expected.get("task")

    task = expected.get("task")
    if task == "parse_write":
        metrics["lane_or_domain_match"] = predicted.get("lane") == expected.get("lane")
        if expected_has_any(expected, "disposition"):
            metrics["disposition_match"] = predicted.get("disposition") == expected.get("disposition")
        if expected_has_any(expected, "reason_code"):
            metrics["reason_code_match"] = predicted.get("reason_code") == expected.get("reason_code")
        expected_records = expected.get("records", [])
        predicted_records = predicted.get("records")
        metrics["record_count_match"] = (
            isinstance(predicted_records, list) and len(predicted_records) == len(expected_records)
        )
        if expected_records and isinstance(predicted_records, list):
            if any("date" in record for record in expected_records):
                metrics["write_dates_match"] = compare_record_field(expected_records, predicted_records, "date")
            if any("unit_text" in record for record in expected_records):
                metrics["unit_text_match"] = compare_record_field(expected_records, predicted_records, "unit_text")
        lane = expected.get("lane")
        if lane == "ledger":
            field_name = "action" if any("action" in r for r in expected_records) else "direction"
            metrics["ledger_action_match"] = compare_record_field(expected_records, predicted_records, field_name)
            metrics["amounts_match"] = compare_record_field(expected_records, predicted_records, "amount")
        elif lane == "expense":
            metrics["amounts_match"] = compare_record_field(expected_records, predicted_records, "amount")
        elif lane == "weight":
            metrics["weight_values_match"] = compare_record_field(expected_records, predicted_records, "value")
    else:
        # parse_query / parse_followup_query. v2 rows carry a uniform field set
        # (disposition, reason_code, clarify_reason, clarify_options) across
        # accept / clarify / reject; v1 rows do not have these. We score each
        # new field only when expected exposes it -> v1 eval rows still pass
        # cleanly through this branch.
        metrics["lane_or_domain_match"] = predicted.get("domain") == expected.get("domain")
        metrics["intent_match"] = predicted.get("intent") == expected.get("intent")
        metrics["query_date_range_match"] = compare_query_range(expected, predicted, "date_start", "date_end")
        metrics["compare_range_match"] = compare_query_range(
            expected, predicted, "compare_date_start", "compare_date_end"
        )
        metrics["filters_match"] = predicted.get("filters") == expected.get("filters")
        metrics["limit_match"] = predicted.get("limit") == expected.get("limit")
        metrics["query_text_match"] = predicted.get("query_text") == expected.get("query_text")
        if expected_has_any(expected, "disposition"):
            metrics["disposition_match"] = predicted.get("disposition") == expected.get("disposition")
        if expected_has_any(expected, "reason_code"):
            metrics["reason_code_match"] = predicted.get("reason_code") == expected.get("reason_code")
        if expected_has_any(expected, "clarify_reason"):
            metrics["clarify_reason_match"] = predicted.get("clarify_reason") == expected.get("clarify_reason")
        if expected_has_any(expected, "clarify_options"):
            metrics["clarify_options_match"] = predicted.get("clarify_options") == expected.get("clarify_options")
        if task == "parse_followup_query":
            metrics["inherit_context_match"] = predicted.get("inherit_context") == expected.get("inherit_context")
    return metrics


def format_rate(correct, applicable):
    if not applicable:
        return None
    return round((correct / applicable) * 100.0, 2)


def summarize_rows(rows):
    summary = {"total_cases": len(rows), "metrics": {}, "task_breakdown": {}}
    metric_names = [
        "valid_json", "exact_match", "task_match", "lane_or_domain_match",
        "disposition_match", "reason_code_match", "clarify_reason_match", "clarify_options_match",
        "intent_match", "record_count_match", "ledger_action_match", "amounts_match",
        "weight_values_match", "write_dates_match", "unit_text_match",
        "query_date_range_match", "compare_range_match", "filters_match",
        "limit_match", "query_text_match", "inherit_context_match",
    ]
    for metric_name in metric_names:
        applicable = 0
        correct = 0
        for row in rows:
            value = row["metrics"][metric_name]
            if value is None:
                continue
            applicable += 1
            if value:
                correct += 1
        summary["metrics"][metric_name] = {
            "correct": correct,
            "applicable": applicable,
            "rate_percent": format_rate(correct, applicable),
        }
    by_task = {}
    for row in rows:
        task = row["expected"]["task"]
        by_task.setdefault(task, {"total": 0, "exact_match": 0, "valid_json": 0})
        by_task[task]["total"] += 1
        if row["metrics"]["exact_match"]:
            by_task[task]["exact_match"] += 1
        if row["metrics"]["valid_json"]:
            by_task[task]["valid_json"] += 1
    for task, counts in by_task.items():
        summary["task_breakdown"][task] = {
            "total": counts["total"],
            "valid_json": counts["valid_json"],
            "valid_json_rate_percent": format_rate(counts["valid_json"], counts["total"]),
            "exact_match": counts["exact_match"],
            "exact_match_rate_percent": format_rate(counts["exact_match"], counts["total"]),
        }
    return summary


def print_summary(label, summary):
    print(f"\n=== {label} ===")
    print(f"Total cases: {summary['total_cases']}")
    preferred_metrics = [
        "valid_json", "exact_match", "task_match", "lane_or_domain_match",
        "disposition_match", "reason_code_match", "clarify_reason_match", "clarify_options_match",
        "intent_match", "record_count_match", "ledger_action_match", "amounts_match",
        "weight_values_match", "write_dates_match", "unit_text_match",
        "query_date_range_match", "compare_range_match", "filters_match",
        "inherit_context_match",
    ]
    for metric_name in preferred_metrics:
        metric = summary["metrics"][metric_name]
        if metric["applicable"] == 0:
            continue
        print(f"{metric_name}: {metric['correct']}/{metric['applicable']} ({metric['rate_percent']}%)")
    print()
    print("task breakdown:")
    for task, counts in summary["task_breakdown"].items():
        print(f"  {task}: total={counts['total']} valid_json={counts['valid_json']}({counts['valid_json_rate_percent']}%) exact_match={counts['exact_match']}({counts['exact_match_rate_percent']}%)")


def save_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
# ---------- LOAD EVAL CASES ----------
cases = load_cases(DATASET)
if LIMIT is not None:
    cases = cases[:LIMIT]
print(f"Loaded {len(cases)} eval cases from {DATASET}")
anchor_present = sum(1 for c in cases if c.get("anchor_date"))
print(f"  carrying anchor_date (v2 rows): {anchor_present}/{len(cases)}")

In [ ]:
# ---------- LOAD MODEL + ADAPTER ----------
def evaluate_one_model(label, model_name_or_adapter, base_model_name, cases, max_seq_length, max_new_tokens, hf_token):
    print(f"\nLoading model for {label}: {model_name_or_adapter}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    adapter_path = Path(model_name_or_adapter)
    use_adapter_loading = (
        label == "finetuned"
        and base_model_name is not None
        and adapter_path.exists()
        and (adapter_path / "adapter_config.json").exists()
    )

    if use_adapter_loading:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
            load_in_8bit=False,
            full_finetuning=False,
            token=hf_token,
        )
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
    else:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=str(adapter_path) if adapter_path.exists() else model_name_or_adapter,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
            load_in_8bit=False,
            full_finetuning=False,
            token=hf_token,
        )
    FastLanguageModel.for_inference(model)

    rows = []
    for i, case in enumerate(cases):
        user_text = build_user_text(case)
        messages = [
            {"role": "system", "content": build_system_prompt(case.get("anchor_date"))},
            {"role": "user", "content": user_text},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()
        parsed_output, parse_error = safe_json_loads(generated)
        metrics = score_prediction(case["expected"], parsed_output)
        rows.append({
            "id": case["id"],
            "anchor_date": case.get("anchor_date"),
            "input": case["input"],
            "context": case.get("context"),
            "expected": case["expected"],
            "raw_output": generated,
            "parsed_output": parsed_output,
            "parse_error": parse_error,
            "metrics": metrics,
        })
        if (i + 1) % 25 == 0 or (i + 1) == len(cases):
            print(f"  scored {i + 1}/{len(cases)}")

    summary = summarize_rows(rows)
    output_dir = Path(OUTPUT_DIR) / label
    output_dir.mkdir(parents=True, exist_ok=True)
    save_jsonl(output_dir / "predictions.jsonl", rows)
    with (output_dir / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print_summary(label, summary)

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, rows

In [ ]:
# ---------- RUN EVAL ----------
summaries = {}
all_rows = {}
if BASE_MODEL is not None:
    summaries["base"], all_rows["base"] = evaluate_one_model(
        "base", BASE_MODEL, None, cases, MAX_SEQ_LENGTH, MAX_NEW_TOKENS, HF_TOKEN,
    )
if FINETUNED_MODEL is not None:
    summaries["finetuned"], all_rows["finetuned"] = evaluate_one_model(
        "finetuned", FINETUNED_MODEL, FINETUNED_BASE_MODEL, cases, MAX_SEQ_LENGTH, MAX_NEW_TOKENS, HF_TOKEN,
    )

combined_path = Path(OUTPUT_DIR) / "combined_summary.json"
combined_path.parent.mkdir(parents=True, exist_ok=True)
with combined_path.open("w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)
print(f"\nSaved combined summary to: {combined_path}")

In [ ]:
# ---------- TOP EXACT-MATCH FAILURES (fine-tuned) ----------
if "finetuned" in all_rows:
    failures = [r for r in all_rows["finetuned"] if not r["metrics"]["exact_match"]]
    print(f"Total exact-match failures: {len(failures)} / {len(all_rows['finetuned'])}")
    for r in failures[:8]:
        print()
        print(f"--- {r['id']}  (anchor={r.get('anchor_date')}) ---")
        print(f"input:    {r['input']}")
        if r.get('context'):
            ctx = json.dumps(r['context'], ensure_ascii=False)
            print(f"context:  {ctx[:200]}")
        print(f"expected: {json.dumps(r['expected'], ensure_ascii=False, separators=(',', ':'))}")
        print(f"raw_out:  {r['raw_output']}")
        if r.get('parse_error'):
            print(f"parse_err:{r['parse_error']}")

## Output

- `/kaggle/working/eval_run/finetuned/predictions.jsonl` â€” full per-row predictions + per-row metrics for the fine-tuned adapter.
- `/kaggle/working/eval_run/finetuned/summary.json` â€” aggregated metrics + per-task breakdown.
- `/kaggle/working/eval_run/base/...` â€” same, for the base model (only if `BASE_MODEL` is set).
- `/kaggle/working/eval_run/combined_summary.json` â€” base + fine-tuned summaries side-by-side.

**Download:** Right sidebar â†’ "Output" tab â†’ drill into `eval_run/`.

**Smoke first, then full:** set `LIMIT = 20` in the CONFIG cell to verify the pipeline end-to-end on 20 rows in a couple of minutes, then re-run with `LIMIT = None` for the full eval. The notebook saves outputs incrementally per model, so even an interrupted run leaves usable per-model artifacts.

**Headline metrics to watch for the v2 adapter:**
- `valid_json` should be ~100%.
- `exact_match` is the strictest metric and the main signal.
- `disposition_match` / `clarify_reason_match` / `clarify_options_match` / `reason_code_match` are v2-specific and only score on rows where `expected` carries those keys (i.e., v2 schema rows).
- `query_date_range_match` and `compare_range_match` test that the model uses the `Today: <anchor_date>` line correctly to resolve relative dates.